# KPSS CANAVARI - Embedding Oluşturma

Bu notebook chunk'lanmış metinlerden embedding'ler oluşturacak.

## Adımlar:
1. Önceki notebook'tan chunk'ları yükle
2. Türkçe embedding modeli yükle
3. Batch processing ile embedding'leri oluştur
4. Embedding'leri kaydet

## Model Seçenekleri:
- `intfloat/multilingual-e5-large` (Önerilir - en iyi performans)
- `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`
- `dbmdz/bert-base-turkish-cased`

## Gereksinimler:
- GPU Runtime (T4 veya daha iyi)
- Tahmini süre: 3-6 saat

In [ ]:
# 1. GPU KONTROLÜ
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Kullanılan cihaz: {device}')

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'📊 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️ GPU bulunamadı! Runtime > Change runtime type > GPU seçin')

In [ ]:
# 2. KÜTÜPHANELERI YÜKLE
!pip install -q sentence-transformers transformers torch tqdm numpy

print('✅ Kütüphaneler yüklendi!')

In [ ]:
# 3. GOOGLE DRIVE BAĞLA VE CHUNK'LARI YÜKLE
from google.colab import drive
import json
import os

drive.mount('/content/drive')

# Yollar
OUTPUT_PATH = '/content/drive/MyDrive/KPSS_Processed'
CHUNKS_FILE = os.path.join(OUTPUT_PATH, 'chunks.json')

# Chunk'ları yükle
print('📥 Chunk\'lar yükleniyor...')
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'✅ {len(chunks):,} chunk yüklendi!')
print(f'\nÖrnek chunk:')
print(chunks[0]['text'][:200] + '...')

In [ ]:
# 4. EMBEDDING MODELİNİ YÜKLE
from sentence_transformers import SentenceTransformer

# Model seç (en iyi performans için)
MODEL_NAME = 'intfloat/multilingual-e5-large'

print(f'📦 Model yükleniyor: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME, device=device)

# Model bilgileri
embedding_dim = model.get_sentence_embedding_dimension()
print(f'✅ Model yüklendi!')
print(f'📏 Embedding boyutu: {embedding_dim}')
print(f'🖥️ Model device: {model.device}')

In [ ]:
# 5. CHECKPOINT SİSTEMİ
import pickle
import numpy as np

EMBEDDINGS_CHECKPOINT = os.path.join(OUTPUT_PATH, 'embeddings_checkpoint.pkl')

def save_embeddings_checkpoint(embeddings, last_index):
    """Embedding checkpoint kaydet"""
    checkpoint = {
        'embeddings': embeddings,
        'last_index': last_index
    }
    with open(EMBEDDINGS_CHECKPOINT, 'wb') as f:
        pickle.dump(checkpoint, f)
    print(f'💾 Checkpoint kaydedildi: {last_index + 1}/{len(chunks)} chunk')

def load_embeddings_checkpoint():
    """Embedding checkpoint yükle"""
    if os.path.exists(EMBEDDINGS_CHECKPOINT):
        with open(EMBEDDINGS_CHECKPOINT, 'rb') as f:
            checkpoint = pickle.load(f)
        print(f'📥 Checkpoint yüklendi: {checkpoint["last_index"] + 1} chunk mevcut')
        return checkpoint['embeddings'], checkpoint['last_index']
    return [], -1

print('✅ Checkpoint sistemi hazır!')

In [ ]:
# 6. EMBEDDING'LERİ OLUŞTUR
from tqdm import tqdm
import time

# Checkpoint'ten devam et
embeddings, last_index = load_embeddings_checkpoint()

# Batch settings
BATCH_SIZE = 32  # GPU memory'ye göre ayarlayın (daha büyük GPU için artırabilirsiniz)
CHECKPOINT_INTERVAL = 1000  # Her 1000 chunk'ta bir checkpoint

print(f'🚀 Embedding oluşturma başlıyor...')
print(f'📦 Batch size: {BATCH_SIZE}')
print(f'💾 Checkpoint interval: {CHECKPOINT_INTERVAL}')

start_time = time.time()
start_index = last_index + 1

# Batch processing
for i in tqdm(range(start_index, len(chunks), BATCH_SIZE), desc='Embedding'):
    # Batch oluştur
    batch_end = min(i + BATCH_SIZE, len(chunks))
    batch_texts = [chunks[j]['text'] for j in range(i, batch_end)]
    
    try:
        # Embedding'leri oluştur
        batch_embeddings = model.encode(
            batch_texts,
            convert_to_numpy=True,
            show_progress_bar=False,
            batch_size=BATCH_SIZE
        )
        
        # Listeye ekle
        for emb in batch_embeddings:
            embeddings.append(emb)
        
        # Checkpoint kaydet
        if (i + BATCH_SIZE) % CHECKPOINT_INTERVAL == 0:
            save_embeddings_checkpoint(embeddings, batch_end - 1)
            
            # İlerleme bilgisi
            elapsed = time.time() - start_time
            progress = len(embeddings) / len(chunks)
            eta = (elapsed / progress - elapsed) if progress > 0 else 0
            print(f'⏱️ Geçen: {elapsed/60:.1f} dk | Kalan: {eta/60:.1f} dk')
    
    except Exception as e:
        print(f'❌ Hata (batch {i}-{batch_end}): {str(e)}')
        # Hataya rağmen checkpoint kaydet
        save_embeddings_checkpoint(embeddings, i - 1)
        break

# Final checkpoint
save_embeddings_checkpoint(embeddings, len(chunks) - 1)

elapsed = time.time() - start_time
print(f'\n✅ Embedding oluşturma tamamlandı!')
print(f'📊 Toplam embedding: {len(embeddings):,}')
print(f'⏱️ Toplam süre: {elapsed/60:.1f} dakika')
print(f'📏 Embedding boyutu: {embeddings[0].shape}')

In [ ]:
# 7. EMBEDDING'LERİ KAYDET
import numpy as np

# NumPy array'e çevir
embeddings_array = np.array(embeddings)

# NPY formatında kaydet (verimli)
embeddings_file = os.path.join(OUTPUT_PATH, 'embeddings.npy')
np.save(embeddings_file, embeddings_array)

print(f'💾 Embedding\'ler kaydedildi: {embeddings_file}')
print(f'📊 Dosya boyutu: {os.path.getsize(embeddings_file) / 1024**3:.2f} GB')

# Metadata kaydet
metadata = {
    'model_name': MODEL_NAME,
    'embedding_dimension': embedding_dim,
    'total_embeddings': len(embeddings),
    'total_chunks': len(chunks),
    'processing_time_minutes': elapsed / 60
}

metadata_file = os.path.join(OUTPUT_PATH, 'embeddings_metadata.json')
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'📋 Metadata kaydedildi: {metadata_file}')

In [ ]:
# 8. TEST: BENZER CHUNK BULMA
from sklearn.metrics.pairwise import cosine_similarity

# Test sorgusu
test_query = "Türkiye Cumhuriyeti'nin başkenti neresidir?"

print(f'🔍 Test sorgusu: "{test_query}"')

# Sorgu embedding'i
query_embedding = model.encode([test_query], convert_to_numpy=True)

# Cosine similarity hesapla
similarities = cosine_similarity(query_embedding, embeddings_array)[0]

# En benzer 5 chunk
top_5_indices = similarities.argsort()[-5:][::-1]

print(f'\n📊 En benzer 5 chunk:')
for i, idx in enumerate(top_5_indices):
    print(f'\n{i+1}. Benzerlik: {similarities[idx]:.4f}')
    print(f'   Kaynak: {chunks[idx]["source_file"]}')
    print(f'   Metin: {chunks[idx]["text"][:150]}...')

print('\n✅ Test başarılı!')

## ✅ TAMAMLANDI!

Sonraki adım: `03_vector_db_upload.ipynb` notebook'unu çalıştırarak Vector Database'e yükleyin.